# YouRA Ablation Study — Overall-Score Statistics

Computes the **mean and sample standard deviation** of the five MLR-Bench rubric
metrics (Clarity, Novelty, Soundness, Significance, Overall) for every ablation
lane under `results/evaluations/mlrbench_overall_score/youra_ablation_study/`.

Aggregation matches Table 1 of the paper: scores are first averaged over the
four judges within each task, then mean ± sample SD is taken across the tasks.

**How to run** — from a fresh clone of this repository, just run all cells.
The first cell regenerates the per-lane CSVs from the raw judge JSONs by calling
`build_ablation_score_stats.py`; the remaining cells recompute the statistics
directly from those CSVs. Only the Python standard library is required
(`pandas` is used for prettier tables when available).

> **Note (2026-07-14):** the `sonnet45_no_IC` lane is now complete: all
> **10 tasks** are collected and reviewed (the `iclr2025_verifai` generation
> finished on 2026-07-14 and its judge reviews were added the same day), so
> every lane below covers the same 10-task set.


In [1]:
from pathlib import Path
import subprocess, sys

def find_script():
    """Locate build_ablation_score_stats.py from any working directory inside the repo."""
    for base in [Path.cwd()] + list(Path.cwd().parents):
        for cand in (base / "build_ablation_score_stats.py",
                     base / "analysis" / "MLRbench_scores_analysis" / "build_ablation_score_stats.py"):
            if cand.is_file():
                return cand
    raise FileNotFoundError(
        "build_ablation_score_stats.py not found - run this notebook from inside the YouRA repository")

SCRIPT = find_script()
STATS_DIR = SCRIPT.parent / "ablation_score_stats"

# Regenerate the CSVs from the raw judge JSONs (works on a fresh clone).
run = subprocess.run([sys.executable, str(SCRIPT)], capture_output=True, text=True)
print(run.stdout)
if run.returncode != 0:
    raise RuntimeError(run.stderr)

lane                          Clarity        Novelty      Soundness   Significance        Overall
sonnet45_no_IC          7.22 +/- 0.18  4.80 +/- 1.13  2.25 +/- 1.01  2.88 +/- 0.44  2.38 +/- 0.44
sonnet45_no_VSA         7.17 +/- 0.46  4.12 +/- 1.09  2.73 +/- 1.29  3.02 +/- 0.92  2.88 +/- 0.94
sonnet45_no_mcp         7.45 +/- 0.42  4.55 +/- 0.90  2.40 +/- 0.64  3.15 +/- 0.39  2.83 +/- 0.54
sonnet45_no_reflection  7.58 +/- 0.41  4.25 +/- 1.05  3.15 +/- 0.95  3.12 +/- 0.58  3.08 +/- 0.68
sonnet46_no_mcp         7.53 +/- 0.30  5.75 +/- 0.60  4.12 +/- 1.21  4.67 +/- 0.95  4.42 +/- 0.80
sonnet46_no_reflection  7.60 +/- 0.29  5.88 +/- 0.73  4.28 +/- 1.26  4.80 +/- 0.77  4.42 +/- 0.94

Wrote 6 task-level CSV(s) and youra_ablation_study_summary.csv to analysis/MLRbench_scores_analysis/ablation_score_stats/



## Load the task-level CSVs

Each `<lane>_task_level_scores.csv` holds one row per *(task, judge)* with the
raw 1–10 scores (plus a precomputed `MEAN(judges)` row per task, which we skip
here and recompute ourselves).

In [2]:
import csv
from collections import defaultdict

METRICS = ["clarity", "novelty", "soundness", "significance", "overall"]

def load_task_csv(path):
    """Return {task: {judge: {metric: score}}} from one task-level CSV."""
    lane = defaultdict(dict)
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if row["judge"] == "MEAN(judges)":
                continue  # recomputed below
            lane[row["task"]][row["judge"]] = {m: float(row[m]) for m in METRICS}
    return dict(lane)

lane_files = sorted(STATS_DIR.glob("*_task_level_scores.csv"))
lanes = {p.name[: -len("_task_level_scores.csv")]: load_task_csv(p) for p in lane_files}
for name, lane in lanes.items():
    n_reviews = sum(len(j) for j in lane.values())
    print(f"{name:24s} {len(lane):2d} tasks  {n_reviews:3d} judge reviews")

sonnet45_no_IC           10 tasks   40 judge reviews
sonnet45_no_VSA          10 tasks   40 judge reviews
sonnet45_no_mcp          10 tasks   40 judge reviews
sonnet45_no_reflection   10 tasks   40 judge reviews
sonnet46_no_mcp          10 tasks   40 judge reviews
sonnet46_no_reflection   10 tasks   40 judge reviews


## Mean ± SD per lane (Table-1 aggregation)

Per metric: average the judges within each task, then take the mean and sample
SD across tasks.

In [3]:
from statistics import mean, stdev

def lane_stats(lane):
    """{metric: (mean_across_tasks, sample_sd_across_tasks)} of judge-averaged task means."""
    out = {}
    for m in METRICS:
        task_means = [mean(s[m] for s in judges.values()) for judges in lane.values()]
        out[m] = (mean(task_means), stdev(task_means) if len(task_means) >= 2 else 0.0)
    return out

summary = {name: lane_stats(lane) for name, lane in lanes.items()}

rows = [{"lane": name, **{f"{m}_mean": round(s[m][0], 4) for m in METRICS},
         **{f"{m}_sd": round(s[m][1], 4) for m in METRICS}}
        for name, s in summary.items()]

try:
    import pandas as pd
    df = pd.DataFrame(rows).set_index("lane")
    display(df[[c for m in METRICS for c in (f"{m}_mean", f"{m}_sd")]])
except ImportError:
    w = max(len(n) for n in summary)
    print(f"{'lane':{w}s}  " + "  ".join(f"{m:>13s}" for m in METRICS))
    for name, s in summary.items():
        print(f"{name:{w}s}  " + "  ".join(f"{s[m][0]:.2f} +/- {s[m][1]:.2f}".rjust(13) for m in METRICS))

,clarity_mean,clarity_sd,novelty_mean,novelty_sd,soundness_mean,soundness_sd,significance_mean,significance_sd,overall_mean,overall_sd
lane,,,,,,,,,,
sonnet45_no_IC,7.225,0.1845,4.800,1.1292,2.250,1.0138,2.875,0.4449,2.375,0.4449
sonnet45_no_VSA,7.175,0.4572,4.125,1.0881,2.725,1.2934,3.025,0.9238,2.875,0.9446
sonnet45_no_mcp,7.450,0.4216,4.550,0.8960,2.400,0.6368,3.150,0.3944,2.825,0.5407
sonnet45_no_reflection,7.575,0.4091,4.250,1.0475,3.150,0.9516,3.125,0.5803,3.075,0.6775
sonnet46_no_mcp,7.525,0.2993,5.750,0.6009,4.125,1.2091,4.675,0.9505,4.425,0.7997
sonnet46_no_reflection,7.600,0.2934,5.875,0.7289,4.275,1.2553,4.800,0.7710,4.425,0.9358


In [4]:
# Human-readable "mean +/- sd" view
w = max(len(n) for n in summary)
print(f"{'lane':{w}s}  " + "  ".join(f"{m.capitalize():>14s}" for m in METRICS))
for name, s in summary.items():
    print(f"{name:{w}s}  " + "  ".join(f"{s[m][0]:.2f} +/- {s[m][1]:.2f}".rjust(14) for m in METRICS))

lane                           Clarity         Novelty       Soundness    Significance         Overall
sonnet45_no_IC           7.22 +/- 0.18   4.80 +/- 1.13   2.25 +/- 1.01   2.88 +/- 0.44   2.38 +/- 0.44
sonnet45_no_VSA          7.17 +/- 0.46   4.12 +/- 1.09   2.73 +/- 1.29   3.02 +/- 0.92   2.88 +/- 0.94
sonnet45_no_mcp          7.45 +/- 0.42   4.55 +/- 0.90   2.40 +/- 0.64   3.15 +/- 0.39   2.83 +/- 0.54
sonnet45_no_reflection   7.58 +/- 0.41   4.25 +/- 1.05   3.15 +/- 0.95   3.12 +/- 0.58   3.08 +/- 0.68
sonnet46_no_mcp          7.53 +/- 0.30   5.75 +/- 0.60   4.12 +/- 1.21   4.67 +/- 0.95   4.42 +/- 0.80
sonnet46_no_reflection   7.60 +/- 0.29   5.88 +/- 0.73   4.28 +/- 1.26   4.80 +/- 0.77   4.42 +/- 0.94


## Cross-check against the script's summary CSV

The notebook's independently recomputed statistics must match
`youra_ablation_study_summary.csv` exactly.

In [5]:
with open(STATS_DIR / "youra_ablation_study_summary.csv", newline="", encoding="utf-8") as f:
    script_summary = {row["lane"]: row for row in csv.DictReader(f)}

mismatches = 0
for name, s in summary.items():
    for m in METRICS:
        for kind, idx in (("mean", 0), ("sd", 1)):
            got, expected = round(s[m][idx], 4), float(script_summary[name][f"{m}_{kind}"])
            if abs(got - expected) > 1e-9:
                mismatches += 1
                print(f"MISMATCH {name} {m}_{kind}: notebook={got} csv={expected}")
print("OK - notebook statistics match the summary CSV exactly." if mismatches == 0
      else f"{mismatches} mismatches found!")

OK - notebook statistics match the summary CSV exactly.


## Per-judge mean ± SD (judge-strictness view)

Same statistics, but per judge across tasks — useful for seeing how much of a
lane's score is driven by individual judges.

In [6]:
for name, lane in lanes.items():
    judges = sorted({j for js in lane.values() for j in js})
    print(f"\n=== {name} ===")
    w = max(len(j) for j in judges)
    print(f"{'judge':{w}s}  " + "  ".join(f"{m.capitalize():>14s}" for m in METRICS))
    for j in judges:
        stats = []
        for m in METRICS:
            vals = [js[j][m] for js in lane.values() if j in js]
            stats.append(f"{mean(vals):.2f} +/- {stdev(vals) if len(vals) >= 2 else 0.0:.2f}")
        print(f"{j:{w}s}  " + "  ".join(c.rjust(14) for c in stats))


=== sonnet45_no_IC ===
judge                          Clarity         Novelty       Soundness    Significance         Overall
claude-opus-4.6          6.10 +/- 0.32   3.30 +/- 1.06   2.30 +/- 0.48   2.30 +/- 0.67   2.00 +/- 0.47
gemini-3.1-pro-preview   8.10 +/- 0.32   5.60 +/- 2.01   1.70 +/- 1.89   1.90 +/- 0.57   1.30 +/- 0.48
gpt-5.4                  6.80 +/- 0.42   5.20 +/- 0.63   2.40 +/- 0.52   4.20 +/- 0.42   3.30 +/- 0.48
grok-4.3                 7.90 +/- 0.57   5.10 +/- 1.37   2.60 +/- 1.58   3.10 +/- 0.88   2.90 +/- 0.88

=== sonnet45_no_VSA ===
judge                          Clarity         Novelty       Soundness    Significance         Overall
claude-opus-4.6          6.00 +/- 0.47   2.70 +/- 0.82   2.60 +/- 0.97   2.20 +/- 0.79   2.20 +/- 0.63
gemini-3.1-pro-preview   7.80 +/- 1.03   3.70 +/- 1.70   1.80 +/- 1.87   1.70 +/- 0.48   1.70 +/- 0.67
gpt-5.4                  7.00 +/- 0.47   4.90 +/- 1.20   3.00 +/- 1.25   4.40 +/- 0.84   3.70 +/- 0.82
grok-4.3                